# Research workflow — live public data

Owner: repository founder. Current workflow: 2026-09-20.

Follow **question → capture and pin → timing/quality → prototype → comparison → backtest → handoff**.
This version obtains one new bounded public Bitfinex R0 recording when you **Run All**.
Use the [offline version](research-foundations.ipynb) for synthetic teaching inputs or a retained selection.
Each run creates new ignored local evidence; network or quality failure stops the run without fallback.

This is foreground development data capture, not a deployed strategy, live trading,
continuous service or economic backtest. See [source limits](../docs/data/l3-collectors.md#verified-source-contract)
and [execution/retention](notebook-operations.md#live-versions).

## 1. Frame the research question

The selected Bitfinex BTC/USD visible order window can support a local quantity
imbalance diagnostic. Its bounded depth and local receipt clock cannot establish
historical eligibility, predictive value, queue rank or exchange-wide completeness.
No model, future label or final holdout is opened.

In [1]:
question = {
    "question": "What visible-size imbalance is observable in this bounded received order window?",
    "sample": "one new public Bitfinex R0 recording; first 32 raw positions for analysis",
    "observable": "(visible buy quantity - visible sell quantity) / total visible quantity",
    "horizon": "individual received states; no forward label",
    "comparator": "zero imbalance; no performance claim",
    "stop_if": "capture or publication fails; absent usable states remain unavailable",
    "fitting": "not applicable",
    "holdout": "not opened",
}
question

{'question': 'What visible-size imbalance is observable in this bounded received order window?',
 'sample': 'one new public Bitfinex R0 recording; first 32 raw positions for analysis',
 'observable': '(visible buy quantity - visible sell quantity) / total visible quantity',
 'horizon': 'individual received states; no forward label',
 'comparator': 'zero imbalance; no performance claim',
 'stop_if': 'capture or publication fails; absent usable states remain unavailable',
 'fitting': 'not applicable',
 'holdout': 'not opened'}

### Prepare the installed candidate

Use the pinned kernel, restart, then run in order. The complete installed inventory
and bytes must match this checkout. Identity includes the notebook source and attempt.

In [2]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from fractions import Fraction
from importlib.metadata import distributions
from pathlib import Path
from uuid import uuid4

# third party library imports
import pandas as pd
from IPython.display import display
from msgspec.structs import replace

# project library imports
import market_data
from applications.explore import inspect_eda
from configuration import CaptureConfig
from market_data.adapters.sources import SourceAdapters
from market_data.contracts import CaptureRequest, Order
from market_data.datasets import DatasetPublication
from market_data.query import BookMarketRow, BookQuery
from market_data.storage import RawStore
from research.exploration import (
    BookResearchInput,
    BookSelection,
    ResearchDataset,
    ResearchReader,
    ResearchSelection,
)
from runtime import ImplementationIdentity, RuntimeFiles
from serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/live/bitfinex-btc-usd.json").is_file(), "Use the repository root/kernel"
assert platform.python_version() == (root / ".python-version").read_text().strip()
assert "site-packages" in Path(market_data.__file__).resolve().parts
codec = DomainCodec()
pd.set_option("display.max_colwidth", None)

In [3]:
source_root = root / "src"
installed_inputs = dict(RuntimeFiles.inputs())
source_files = {
    path.relative_to(source_root).as_posix(): path
    for package in RuntimeFiles.roots
    for path in (source_root / package).rglob("*")
    if path.is_file() and (path.suffix in {".py", ".json"} or path.name == "py.typed")
}
assert source_files.keys() == installed_inputs.keys(), "Installed runtime inventory differs"
for name, source in source_files.items():
    assert source.read_bytes() == installed_inputs[name].read_bytes(), (
        f"Stale installed file: {name}"
    )
notebook_path = Path(
    os.environ.get(
        "ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations-live.ipynb"
    )
)
notebook = json.loads(notebook_path.read_text())
notebook_source = [(cell["cell_type"], "".join(cell["source"])) for cell in notebook["cells"]]
identity = {
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip(),
    "working_tree": "changed"
    if subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
    else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel", "pandas"}
    },
    "kernel": sys.executable,
    "package": market_data.__file__,
    "runtime_files": len(installed_inputs),
    "implementation_sha256": ImplementationIdentity.sha256(),
    "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "notebook_source_sha256": hashlib.sha256(
        json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
    ).hexdigest(),
    "attempt": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
}
display(pd.DataFrame(identity.items(), columns=["Identity", "Value"]))

,Identity,Value
0,commit,00373626acc1459136163297cd4a76d2c0542787
1,working_tree,changed
2,python,3.14.7
3,platform,macOS-27.0-arm64-arm-64bit-Mach-O
4,tooling,"{'nbformat': '5.10.4', 'ipykernel': '7.2.0', 'nbclient': '0.10.4', 'pandas': '3.0.5'}"
5,kernel,/Users/lxg/.cache/uv/builds-v0/.tmp0rEXD3/bin/python
6,package,/private/tmp/alpha-research-live-offline-notebooks/.venv/lib/python3.14/site-packages/market_data/__init__.py
7,runtime_files,138
8,implementation_sha256,b67f60a16bd5f84fc3b0898389638dca4e16a7372418fa5fd3b0ad80c8d9c5f9
9,lock_sha256,e6cd3a169ff4b6850bcb5b09e38c44e1179b0dc1fb7c6a9bd95d2c4a058595e4


## 2. Acquire and pin a bounded sample

The next configuration cell creates an exclusive retained study directory. The capture
cell contacts the configured public feed for at most 20 seconds of receipt, bounded by
messages, bytes, queues, idle time and retries. Completion/fsync can extend elapsed time;
the notebook also applies an 80-second child deadline. A timeout kills/waits for its child
and retains the partial recording for explicit interrupted inspection. No keys are read.
Review the displayed limits before running the capture cell interactively.

In [4]:
config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
if config.venue != "bitfinex" or config.channel != "R0" or config.duration_seconds > 20:
    raise ValueError("This notebook selects public Bitfinex R0 with a receive interval <=20s")
study = root / "data/notebooks/research-live" / str(uuid4())
study.mkdir(parents=True)
(study / "snapshots").mkdir()
(study / "raw").mkdir()
config_path = study / "capture-config.json"
config_path.write_bytes(codec.encode_json(config))
(study / "attempt.json").write_bytes(
    codec.encode_json({"identity": identity, "question": question})
)
display(
    pd.DataFrame(
        [
            {
                "venue": config.venue,
                "symbol": config.symbol,
                "channel": config.channel,
                "base": config.base_currency,
                "quote": config.quote_currency,
                "duration_seconds": config.duration_seconds,
                "messages_max": config.max_messages,
                "journal_bytes_max": config.max_bytes,
                "frame_bytes_max": config.max_frame_bytes,
                "retries": config.retries,
                "published_raw_positions_max": 32,
            }
        ]
    )
)
print("Retained attempt directory:", study)

,venue,symbol,channel,base,quote,duration_seconds,messages_max,journal_bytes_max,frame_bytes_max,retries,published_raw_positions_max
0,bitfinex,tBTCUSD,R0,BTC,USD,20,10000,32000000,8000000,2,32


Retained attempt directory: /private/tmp/alpha-research-live-offline-notebooks/data/notebooks/research-live/1c256199-49d0-4bdd-9f21-993c461a75b7


In [5]:
collector = Path(sys.executable).parent / "alpha-collect"
if not collector.is_file():
    raise FileNotFoundError(f"Installed collector is missing: {collector}")
command = [
    str(collector),
    "capture",
    "--config",
    str(config_path),
    "--output",
    str(study / "captures"),
]
# Claim this attempt before any source effect; rerunning this cell requires new setup.
with (study / "command.json").open("xb") as command_file:
    command_file.write(codec.encode_json(command))
# Files retain partial diagnostics even if the child times out; no raw frames are displayed.
with (study / "capture.stdout").open("x") as stdout, (study / "capture.stderr").open("x") as stderr:
    capture = subprocess.run(
        command, cwd=study, stdout=stdout, stderr=stderr, timeout=80, check=False
    )
(study / "capture-exit.json").write_bytes(codec.encode_json({"exit_code": capture.returncode}))
if capture.returncode != 0:
    raise RuntimeError(
        f"Capture exit {capture.returncode}; inspect retained logs/prefix under {study}"
    )
summary = json.loads((study / "capture.stdout").read_text())
capture_directory = Path(summary["capture_directory"])
request = codec.decode_json((capture_directory / "request.json").read_bytes(), CaptureRequest)
assert request.config == config and request.identity == summary["request_identity"]
display(
    pd.DataFrame(
        [
            {
                name: summary[name]
                for name in (
                    "venue",
                    "symbol",
                    "messages",
                    "epochs",
                    "gaps",
                    "stop_reason",
                    "replay_parity",
                )
            }
        ]
    )
)

,venue,symbol,messages,epochs,gaps,stop_reason,replay_parity
0,bitfinex,tBTCUSD,398,1,[],duration_limit,True


Publication and input selection are separate explicit operations after successful capture.
The shared publisher verifies saved originals and admits only the first 32 raw positions;
that prefix may contain controls and fewer than 32 market states. One immutable selection
names its snapshot, request, fields and receipt/cutoff window. No latest-input search occurs.

In [6]:
# Copy the verified original into this study's raw/ store and publish a bounded prefix.
snapshot_id = DatasetPublication(SourceAdapters(), ImplementationIdentity.sha256()).publish_books(
    (capture_directory,),
    RawStore(study / "raw"),
    study / "snapshots",
    max_records=32,
    max_bytes=config.max_bytes,
)
# Query boundaries are microseconds; floor the start and ceil the excluded end.
# Original per-message integer nanoseconds remain untouched in the result.
epoch = datetime(1970, 1, 1, tzinfo=UTC)
start = epoch + timedelta(microseconds=summary["started_ns"] // 1000)
end = epoch + timedelta(microseconds=summary["finished_ns"] // 1000 + 1)
selection = ResearchSelection(
    inputs=(
        BookSelection(
            name="visible_depth",
            query=BookQuery(
                snapshot_id=snapshot_id,
                request_identity=request.identity,
                instrument=config.instrument_id,
                start=start,
                end=end,
                known_at=end,
                columns=("orders",),
                batch_size=32,
                max_bytes=config.max_bytes,
            ),
        ),
    )
)
selection_path = study / "selection.json"
selection_path.write_bytes(codec.encode_json(selection))
print("Snapshot:", snapshot_id)
print("Selection:", selection.identity)
print("UTC receipt interval [start, end):", start, end)

Snapshot: 586b6711340d04ff8575049b65fe4112f84b16be01ad005c7dcefae26a1e14c0
Selection: 65bcaaff2403f0f7e48e3c2163efba55ecb3f26322fa9e8451ad060fae4ac5fc
UTC receipt interval [start, end): 2026-09-19 16:45:43.539732+00:00 2026-09-19 16:46:03.735413+00:00


## 3. Inspect quality, missingness and receipt time

Use the same installed EDA entry as the offline version. Inspect controls beside values.
Book states remain exploratory/degraded, and absent states do not mean an empty market.
The final capture state is unavailable after session end even when reconstruction matched.

In [7]:
class StudyReports:
    """Retain EDA status beside the original live study."""

    @staticmethod
    def retain(record: dict[str, object]) -> None:
        """Append one application status record.

        Args:
            record: Application-owned status and input identities.
        """
        with (study / "eda-attempts.jsonl").open("ab") as output:
            output.write(codec.encode_json(record) + b"\n")


result = inspect_eda(
    selection_path, study / "snapshots", RawStore(study / "raw"), report=StudyReports.retain
)
dataset = result["dataset"]
assert isinstance(dataset, ResearchDataset) and dataset.identity == selection.identity
depth = dataset.inputs[0]
assert isinstance(depth, BookResearchInput)
market_rows = [
    row for batch in depth.batches for row in batch.rows if isinstance(row, BookMarketRow)
]
controls = [
    row for batch in depth.batches for row in batch.rows if not isinstance(row, BookMarketRow)
]
display(
    pd.DataFrame(
        [
            {
                "market_states": len(market_rows),
                "control_boundaries": len(controls),
                "base_quantity_unit": config.base_currency,
                "price_quote_unit": config.quote_currency,
                "coverage_limits": sorted(
                    {reason for batch in depth.batches for reason in batch.reasons}
                ),
            }
        ]
    )
)
display(
    pd.DataFrame(
        [
            {
                "position": row.position,
                "received_ns": row.received_ns,
                "status": row.status,
                "reason": row.reason,
            }
            for row in controls
        ]
    )
)

,market_states,control_boundaries,base_quantity_unit,price_quote_unit,coverage_limits
0,28,4,BTC,USD,[visible depth only; coverage and historical eligibility unproven]


,position,received_ns,status,reason
0,0,1789836343539732000,recovering,validated snapshot required
1,1,1789836343853638000,recovering,validated snapshot required
2,2,1789836344040568000,recovering,validated snapshot required
3,3,1789836344466742000,recovering,validated snapshot required


Move the cutoff before this recording. No subsequently received state may enter that
information set. Then reopen the original selection and compare the complete result.

In [8]:
reader = ResearchReader(study / "snapshots", RawStore(study / "raw"), readers=SourceAdapters())
early = ResearchSelection(
    inputs=(
        replace(
            selection.inputs[0],
            query=replace(selection.inputs[0].query, known_at=start - timedelta(microseconds=1)),
        ),
    )
)
early_dataset = reader.read(early)
assert not any(
    isinstance(row, BookMarketRow)
    for item in early_dataset.inputs
    for batch in item.batches
    for row in batch.rows
)
assert reader.read(selection) == dataset
print("No values before this recording; original pinned selection reproduced.")

No values before this recording; original pinned selection reproduced.


## 4. Prototype a feature; keep its label separate

For each admitted received L3 state, sum exact base quantities on each side and retain
the dimensionless imbalance as an exact rational. Use only these message-local visible
orders; do not forward-fill controls or infer missing depth. Zero total is unavailable.
Market-derived ratios and state identities are kept in the ignored study directory.
A later return label needs separately selected prices, horizon and availability evidence.

In [9]:
prototype = []
for row in market_rows:
    orders = row.values[0]
    assert all(isinstance(order, Order) for order in orders)
    # Fractions preserve exact decimal quantities without depending on ambient Decimal precision.
    bids = sum((Fraction(order.size) for order in orders if order.side == "buy"), Fraction())
    asks = sum((Fraction(order.size) for order in orders if order.side == "sell"), Fraction())
    total = bids + asks
    imbalance = None if total == 0 else (bids - asks) / total
    prototype.append(
        {
            "position": row.position,
            "received_ns": row.received_ns,
            "state": row.state_identity,
            "quality": row.status,
            "imbalance_exact_ratio": None if imbalance is None else str(imbalance),
        }
    )
(study / "prototype.json").write_bytes(codec.encode_json(prototype))
# Market-derived values stay in ignored local evidence; tracked notebook outputs show counts only.
print("Computed exploratory states:", len(prototype))
print("Unavailable denominators:", sum(item["imbalance_exact_ratio"] is None for item in prototype))
print("Exact ratios and state identities retained:", study / "prototype.json")

Computed exploratory states: 28
Unavailable denominators: 0
Exact ratios and state identities retained: /private/tmp/alpha-research-live-offline-notebooks/data/notebooks/research-live/1c256199-49d0-4bdd-9f21-993c461a75b7/prototype.json


## 5. Compare models or deterministic rules — planned

This short live sample is an observation, not model selection or predictive evidence.
Supply a declared hypothesis, temporal splits, comparison/stop criteria and all influencing
attempts before formal research. Rules need no artificial fit stage.

## 6. Backtest and evaluate economics — planned

Economic evaluation still needs a target policy, positive-delay execution, costs,
independent limits and reconciled accounting. Visible orders establish no fills.

## 7. Freeze, qualify and monitor — planned; retain today's handoff

The current handoff is the retained selection, raw/snapshot bytes, configuration,
question, EDA statuses and prototype. Keep every failed/influencing real-data attempt
under the [research record contract](../docs/roadmap/research-program.md#small-records-and-artifacts).
Local ignored files are not a backed-up journal. Do not tune frozen P1 from this sample.
Use the printed directory in the offline QR version; that reopening makes no connection.

In [10]:
handoff = {
    "selection": selection.identity,
    "snapshot": snapshot_id,
    "request": request.identity,
    "capture": str(capture_directory),
    "market_states": len(market_rows),
    "status": "exploratory; no economic acceptance",
}
(study / "handoff.json").write_bytes(codec.encode_json(handoff))
print("Offline QR: set ALPHA_RESEARCH_DATASET_DIRECTORY to", study)
print("Offline engineer: set ALPHA_BITFINEX_CAPTURE_DIRECTORY to", capture_directory)
print("Retained all inputs and attempts; no automatic deletion or repeated capture.")

Offline QR: set ALPHA_RESEARCH_DATASET_DIRECTORY to /private/tmp/alpha-research-live-offline-notebooks/data/notebooks/research-live/1c256199-49d0-4bdd-9f21-993c461a75b7
Offline engineer: set ALPHA_BITFINEX_CAPTURE_DIRECTORY to /private/tmp/alpha-research-live-offline-notebooks/data/notebooks/research-live/1c256199-49d0-4bdd-9f21-993c461a75b7/captures/2c52105f-e805-436c-b0fb-dbd1ebf3cf34
Retained all inputs and attempts; no automatic deletion or repeated capture.
